# Ambil Dataset Jurnal Indonesia + Buat Embedding (Google Colab)

Notebook ini menjalankan SELURUH alur di Colab (yang punya internet), tanpa instalasi di laptop:
1. cek apakah endpoint OAI-PMH jurnal aktif, 2. panen metadata (judul, abstrak, kata kunci, DOI), 3. bangun dataset bersih,
4. buat embedding (e5-small), 5. unduh hasilnya.

**Sebelum mulai**
- Unggah ZIP proyek (`capstone-semantic-retrieval_diperbaiki.zip`) saat diminta di langkah 1.
- Isi email aktif Anda di langkah 3 (dipakai sebagai identitas sopan ke server jurnal).
- Jalankan sel satu per satu dari atas. Perkiraan waktu: 10–40 menit (panen dibatasi 1 permintaan/detik agar tidak membebani server jurnal).
- *Catatan:* notebook ini belum pernah dijalankan langsung oleh pengembang (sandbox tanpa internet). Bila ada sel gagal, salin pesan error-nya.

Hanya **metadata** yang diambil. Lisensi: RESTI CC BY 4.0, JNTETI CC BY-SA 4.0. Baca `data/DATA_SOURCES.md` dan `docs/checklist_publikasi.md` sebelum mempublikasikan data.

## 1. Unggah dan ekstrak proyek

In [ ]:
from google.colab import files
import os, glob
up = files.upload()   # pilih capstone-semantic-retrieval_diperbaiki.zip
zip_name = next(iter(up))
!unzip -q -o "{zip_name}" -d /content
%cd /content/capstone-semantic-retrieval
!ls

## 2. Pasang library

In [ ]:
!pip install -q -r requirements.txt

## 3. Identitas (WAJIB diisi)

In [ ]:
import os
EMAIL = "ISI_EMAIL_ANDA@contoh.ac.id"   # <<< ganti
assert "ISI_EMAIL" not in EMAIL, "Isi EMAIL dengan alamat email aktif Anda."
os.environ["CONTACT_EMAIL"] = EMAIL

## 4. Cek sumber jurnal
Mencari tahu endpoint mana yang benar-benar aktif. Baca keluarannya:
`[OK]` = bisa dipanen. `[GAGAL]`/error = endpoint salah atau diblokir; sumber itu dilewati.

In [ ]:
!python -m scripts.check_sources --update-config --contact-email "{EMAIL}"

## 5. Panen metadata
Setiap jurnal dibatasi 300 artikel (setelah filter topik informatika/AI/NLP di `config/topic_terms.txt`).
Hanya sumber yang lolos langkah 4 (`enabled=true`) yang dipanen.

In [ ]:
!python -m scripts.collect_oai --max-per-source 300 --contact-email "{EMAIL}"

In [ ]:
import glob, pandas as pd
files_ = sorted(glob.glob("data/real/harvest/*.csv"))
print("Berkas hasil panen:", files_)
for f in files_:
    d = pd.read_csv(f)
    print(f, "->", len(d), "baris | abstrak terisi:", int((d.get("Abstract", pd.Series(dtype=str)).fillna("").str.len() > 50).sum()),
          "| kata kunci terisi:", int((d.get("Keywords", pd.Series(dtype=str)).fillna("") != "").sum()))
assert files_, "Tidak ada hasil panen. Lihat keluaran langkah 4-5; lanjut ke 'Jalur cadangan' di bawah."


## 6. Bangun dataset
Hanya memakai hasil panen jurnal (`data/real/harvest`), **bukan** data Publish or Perish berbahasa Inggris.
Dataset sementara sebelumnya dicadangkan dulu.

In [ ]:
import shutil, os
if os.path.isdir("artifacts_real") and not os.path.isdir("artifacts_real_pop_backup"):
    shutil.copytree("artifacts_real", "artifacts_real_pop_backup")
!python -m scripts.build_dataset --kind real --input-dirs data/real/harvest
print(open("results/validation_real.md", encoding="utf-8").read()[:2500])

Periksa laporan di atas. Idealnya: **LOLOS**, jumlah dokumen 200–500, bahasa Indonesia terdeteksi. Bila di bawah 200,
ulangi langkah 5 dengan `--max-per-source` lebih besar atau tambahkan jurnal di `config/journals.csv`.

## 7. Buat dan periksa embedding

In [ ]:
!python -m scripts.generate_embeddings --artifacts-dir artifacts_real
!python -m scripts.check_embeddings --artifacts-dir artifacts_real --sanity

## 8. Unduh hasil
Ekstrak ZIP ke folder proyek di laptop Anda (akan menimpa `artifacts_real/`), lalu jalankan:
`streamlit run app.py` dengan `ARTIFACTS_DIR=artifacts_real`.

In [ ]:
shutil.make_archive("artifacts_real_hasil", "zip", ".", "artifacts_real")
files.download("artifacts_real_hasil.zip")
# Simpan juga hasil panen (untuk bukti sumber data):
shutil.make_archive("harvest_hasil", "zip", ".", "data/real/harvest")
files.download("harvest_hasil.zip")

## Jalur cadangan: bila panen OAI gagal
1. Buka **Publish or Perish**, sumber *Crossref*, cari berdasarkan **ISSN** jurnal (RESTI 2580-0760, JNTETI 2460-5719), simpan sebagai CSV.
2. Unggah CSV ke Colab, lalu:
```
!python -m scripts.import_csv --input FILE.csv --name resti_pop --data-source real --source-note "PoP/Crossref ISSN 2580-0760, <tanggal>" --exclude-types "edited-book,monograph,book,proceedings"
!python -m scripts.derive_keywords --input data/real/imports/resti_pop.csv --output data/real/imports/resti_pop_kw.csv
```
(ganti nama berkas, lalu hapus versi tanpa `_kw` dari `data/real/imports/` sebelum `build_dataset`). Ingat: ekspor PoP/Crossref sering tanpa abstrak, jadi banyak baris akan terbuang.